# Chapter 15 — Building AI Agents: Autonomous Network Reasoning

**Volume 2: Practical Applications — AI for Networking and Security Engineers**

In Chapter 14 you built a RAG system that **answers questions** from documentation.
In this chapter you build an **AI agent** that can **take actions** — SSH to routers,
run diagnostic commands, correlate outputs, and reason toward a conclusion autonomously.

> *In Simple Words: A RAG system is a very smart library. An agent is a junior engineer
> you can delegate to. The library answers questions. The engineer does the work.*

---
**What you will build — 5 hands-on demos:**

| # | Demo | Key concept |
|---|------|-------------|
| 1 | The Agent Loop Visualized | Perception → Reason → Act → Update |
| 2 | Tool Calling with Claude | JSON schema tools + `tool_use` stop reason |
| 3 | ReAct Network Troubleshooting Agent | Thought → Action → Observation loop |
| 4 | Agent Memory Architecture | Working / Semantic / Episodic / Procedural |
| 5 | Multi-Agent: Manager + Specialists | Coordinator routes to BGP/OSPF/Interface agents |

> **Prerequisite:** `!pip install anthropic` (run the install cell first)


In [ ]:
!pip install -q anthropic scikit-learn numpy

# ── API Key Setup ──────────────────────────────────────────────────────────
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")


---
## Demo 1 — The Agent Loop: Perception → Reason → Act → Update

Every AI agent — regardless of complexity or framework — runs **one loop**.
Understanding this loop at the theoretical level unlocks everything else.

```
┌────────────────────────────────────────────────────────────┐
│                     THE AGENT LOOP                         │
│                                                            │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐           │
│   │ PERCEIVE │───▶│  REASON  │───▶│   ACT    │           │
│   └──────────┘    └──────────┘    └──────────┘           │
│         ▲                               │                  │
│         └───────────── UPDATE ◀─────────┘                  │
│                   (tool result)                            │
└────────────────────────────────────────────────────────────┘
```

**The networking analogy:**
- OSPF runs exactly this loop: perceive LSAs → run SPF → update routing table → repeat
- BGP runs this loop: perceive path updates → run best-path selection → advertise → repeat
- The difference: routing protocols use fixed algorithms. AI agents use an LLM for reasoning.

This demo simulates the loop step-by-step so you can see each phase.


In [ ]:
# Demo 1: The Agent Loop — visualized step by step (no API needed)
class AgentLoop:
    """
    A simple agent loop simulator.
    Perception → Reason → Act → Update, repeating until goal achieved.
    """

    def __init__(self, name: str):
        self.name       = name
        self.context    = []   # working memory (what the agent knows so far)
        self.iteration  = 0
        self.max_iter   = 5    # safety limit — always have one!

    def perceive(self, new_info: str):
        """Phase 1: Add new information to working memory."""
        self.context.append(new_info)
        print(f"  👁️  PERCEIVE: {new_info}")

    def reason(self) -> str:
        """Phase 2: Decide what to do next based on current context."""
        # Simplified logic — in a real agent, this is the LLM call
        ctx = " ".join(self.context).lower()
        if "bgp" in ctx and "neighbor" not in ctx:
            return "check_bgp_neighbors"
        elif "neighbor" in ctx and "established" not in ctx:
            return "check_bgp_timers"
        elif "timer" in ctx and "mismatch" in ctx:
            return "fix_bgp_timers"
        else:
            return "report_conclusion"

    def act(self, action: str) -> str:
        """Phase 3: Execute the chosen action (call a tool)."""
        print(f"  ⚡ ACT: {action}()")
        # Simulated tool responses
        responses = {
            "check_bgp_neighbors": "BGP neighbor 10.1.1.2 state: IDLE",
            "check_bgp_timers":    "Local: hold=90s  Remote: hold=180s  → MISMATCH",
            "fix_bgp_timers":      "Applied: neighbor 10.1.1.2 timers 60 180  → Session now ESTABLISHED",
            "report_conclusion":   "ROOT CAUSE: BGP timer mismatch. Hold timer corrected. Session established.",
        }
        return responses.get(action, "Unknown action")

    def update(self, observation: str):
        """Phase 4: Add the result back to working memory."""
        self.context.append(observation)
        print(f"  📥 UPDATE: {observation}")

    def run(self, initial_task: str):
        """Run the full loop until goal is achieved or max iterations reached."""
        print(f"
{'='*60}")
        print(f"  AGENT: {self.name}")
        print(f"  TASK:  {initial_task}")
        print(f"{'='*60}
")

        self.perceive(initial_task)

        while self.iteration < self.max_iter:
            self.iteration += 1
            print(f"
  ── Iteration {self.iteration} ──────────────────────────")

            # Reason
            action = self.reason()
            print(f"  🧠 REASON: decided to → {action}")

            # Act
            observation = self.act(action)

            # Update
            self.update(observation)

            # Terminal condition
            if action == "report_conclusion" or "ESTABLISHED" in observation:
                print(f"
  ✅ GOAL ACHIEVED in {self.iteration} iteration(s)")
                break
        else:
            print(f"
  ⚠️  Max iterations ({self.max_iter}) reached — escalating to human")

        print(f"
  Final context window ({len(self.context)} entries):")
        for i, entry in enumerate(self.context, 1):
            print(f"    [{i}] {entry}")


# Run the agent
agent = AgentLoop(name="BGP-Troubleshooter-v1")
agent.run("BGP session to peer 10.1.1.2 is down — investigate and fix")


---
## Demo 2 — Tool Calling with Claude: How It Really Works

**The key misconception:** The LLM never directly executes code.

**The actual flow:**
```
Your code sends:   [conversation + tool definitions]
Claude returns:    stop_reason = "tool_use" + tool name + arguments
Your code runs:    the actual Python function
Your code sends:   the result back as tool_result
Claude continues:  reasoning with the observation
```

This is like **NETCONF**:
- Claude = NETCONF client (generates RPC requests, never touches the device)
- Your Python code = NETCONF server (executes the operations)
- This separation = where ALL safety controls live

Tools are defined as **JSON schemas** — just a description of what the tool does
and what parameters it takes. Claude reads this and acts as a dispatcher.


In [ ]:
# Demo 2: Tool calling with Claude — see every step clearly
import anthropic
import json

client = anthropic.Anthropic()
HAIKU = "claude-haiku-4-5-20251001"

# ── Step 1: Define tools as JSON schemas ────────────────────────────────────
# These are DESCRIPTIONS — Claude reads them to know what it can do.
# Your Python functions do the actual work.
TOOLS = [
    {
        "name": "ping_device",
        "description": "Ping a network device to check reachability. Returns packet loss % and RTT.",
        "input_schema": {
            "type": "object",
            "properties": {
                "hostname": {
                    "type": "string",
                    "description": "IP address or hostname to ping (e.g., 10.1.1.1 or core-router-01)",
                },
                "count": {
                    "type": "integer",
                    "description": "Number of ICMP packets to send (default 5)",
                },
            },
            "required": ["hostname"],
        },
    },
    {
        "name": "show_interface",
        "description": "Run 'show interface' on a device. Returns status, errors, and traffic counters.",
        "input_schema": {
            "type": "object",
            "properties": {
                "device":    {"type": "string", "description": "Device hostname or IP"},
                "interface": {"type": "string", "description": "Interface name, e.g. GigabitEthernet0/1"},
            },
            "required": ["device", "interface"],
        },
    },
    {
        "name": "show_bgp_summary",
        "description": "Run 'show bgp summary' on a device. Returns neighbor table with states.",
        "input_schema": {
            "type": "object",
            "properties": {
                "device": {"type": "string", "description": "Device hostname or IP"},
            },
            "required": ["device"],
        },
    },
]

# ── Step 2: The actual Python functions (simulated for Colab safety) ─────────
def ping_device(hostname: str, count: int = 5) -> str:
    """Simulated ping — in production, use subprocess + ping command."""
    simulated = {
        "10.1.1.1":       f"PING 10.1.1.1: {count}/{count} packets, 0% loss, RTT avg 1ms",
        "10.1.1.2":       f"PING 10.1.1.2: 0/{count} packets, 100% loss — HOST UNREACHABLE",
        "core-router-01": f"PING core-router-01: {count}/{count} packets, 0% loss, RTT avg 2ms",
    }
    return simulated.get(hostname, f"PING {hostname}: {count}/{count} packets, 0% loss, RTT avg 5ms")

def show_interface(device: str, interface: str) -> str:
    """Simulated show interface output."""
    return (
        f"{interface} is up, line protocol is up
"
        f"  Hardware is Gigabit Ethernet
"
        f"  Input errors: 0, CRC: 0, Overrun: 0
"
        f"  Output errors: 0, Collisions: 0
"
        f"  5 minute input rate: 45,000 bits/sec, 12 packets/sec"
    )

def show_bgp_summary(device: str) -> str:
    """Simulated BGP neighbor table."""
    return (
        f"BGP router identifier 10.0.0.1, local AS number 65001
"
        f"Neighbor        AS      State    PfxRcd
"
        f"10.1.1.2        65002   IDLE     0       ← not established!
"
        f"10.1.1.3        65003   Established  150"
    )

# Tool dispatcher — maps tool names to Python functions
TOOL_REGISTRY = {
    "ping_device":     lambda args: ping_device(**args),
    "show_interface":  lambda args: show_interface(**args),
    "show_bgp_summary": lambda args: show_bgp_summary(**args),
}

# ── Step 3: Single tool-call round-trip (transparent) ────────────────────────
def call_claude_with_tools(task: str, verbose: bool = True) -> str:
    """
    One pass through the tool-calling protocol.
    Returns Claude's final text answer.
    """
    messages = [{"role": "user", "content": task}]
    step = 0

    while True:
        step += 1
        response = client.messages.create(
            model=HAIKU,
            max_tokens=500,
            tools=TOOLS,
            messages=messages,
        )

        # Append Claude's response to conversation history
        messages.append({"role": "assistant", "content": response.content})

        if verbose:
            print(f"
  ── Step {step} ── stop_reason: {response.stop_reason}")

        if response.stop_reason == "tool_use":
            # Find all tool-use blocks in this response
            for block in response.content:
                if block.type == "text" and block.text.strip():
                    print(f"  🧠 Thought: {block.text.strip()[:150]}")
                elif block.type == "tool_use":
                    print(f"  ⚡ Action:  {block.name}({json.dumps(block.input)})")

                    # Execute the tool
                    result = TOOL_REGISTRY[block.name](block.input)
                    print(f"  📥 Result:  {result[:120]}")

                    # Feed result back as tool_result
                    messages.append({
                        "role": "user",
                        "content": [{
                            "type":        "tool_result",
                            "tool_use_id": block.id,
                            "content":     result,
                        }],
                    })

        elif response.stop_reason == "end_turn":
            final = next((b.text for b in response.content if b.type == "text"), "")
            print(f"
  ✅ Final Answer:
  {final}")
            return final

        else:
            print(f"  Unexpected stop_reason: {response.stop_reason}")
            break


# ── Run it ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("  TOOL CALLING DEMO")
print("=" * 60)
call_claude_with_tools(
    "Check if device 10.1.1.1 is reachable and then show me the BGP summary for core-router-01"
)


---
## Demo 3 — ReAct Agent: Autonomous Network Troubleshooting

**ReAct = Reasoning + Acting** — the agent writes its Thought before each Action.
This makes the reasoning **transparent, debuggable, and self-correcting**.

```
Thought: "BGP is down. I should first check if the peer is even reachable at layer 3."
Action:  ping_device(hostname="10.1.1.2")
Observation: "100% packet loss — HOST UNREACHABLE"
Thought: "IP connectivity is lost. The BGP issue is a symptom, not the root cause. Check the interface."
Action:  show_interface(device="core-01", interface="GigabitEthernet0/1")
...
```

The Thought step is not decorative — it:
- **Decomposes** complex problems into sequential sub-tasks
- **Reduces hallucination** by committing to intermediate conclusions
- **Makes debugging possible** — you can read where it went wrong

This agent has 6 simulated network tools covering the full diagnostic path.


In [ ]:
# Demo 3: Full ReAct troubleshooting agent with 6 network tools
import anthropic
import json

client = anthropic.Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

# ── Simulated network state (like a real lab environment) ────────────────────
NETWORK_STATE = {
    "core-01": {
        "bgp_peers": {
            "10.1.1.2": {"state": "IDLE",        "as": 65002, "hold_local": 90,  "hold_remote": 180},
            "10.1.1.3": {"state": "Established",  "as": 65003, "hold_local": 90,  "hold_remote": 90},
        },
        "interfaces": {
            "GigabitEthernet0/0": {"status": "up",   "errors": 0,    "duplex": "full"},
            "GigabitEthernet0/1": {"status": "up",   "errors": 0,    "duplex": "full"},
        },
        "routes": ["10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16"],
        "ospf_neighbors": {"10.2.2.2": "FULL", "10.2.2.3": "FULL"},
        "logs": [
            "BGP-5-ADJCHANGE: neighbor 10.1.1.2 Down -- Hold Timer Expired",
            "BGP-3-NOTIFICATION: received from neighbor 10.1.1.2 code/subcode=4/0",
        ],
    }
}

# ── Tool functions ────────────────────────────────────────────────────────────
def ping_device(device: str, target: str, count: int = 5) -> str:
    reachable = {"10.1.1.2": True, "10.1.1.3": True, "10.0.0.1": True}
    ok = reachable.get(target, True)
    if ok:
        return f"Success: {count}/{count} packets, 0% loss, avg RTT 1ms"
    return f"FAILED: 0/{count} packets, 100% loss — unreachable"

def show_bgp_summary(device: str) -> str:
    if device not in NETWORK_STATE:
        return f"Error: device {device} not found"
    peers = NETWORK_STATE[device]["bgp_peers"]
    lines = [f"BGP router: {device} | Local AS: 65001", "-" * 45]
    for ip, info in peers.items():
        lines.append(f"  {ip:15} AS {info['as']}  State: {info['state']}")
    return "
".join(lines)

def show_bgp_neighbor_detail(device: str, neighbor_ip: str) -> str:
    if device not in NETWORK_STATE:
        return f"Error: device {device} not found"
    peer = NETWORK_STATE[device]["bgp_peers"].get(neighbor_ip)
    if not peer:
        return f"Neighbor {neighbor_ip} not found in BGP table"
    return (
        f"BGP neighbor {neighbor_ip}, remote AS {peer['as']}, external link
"
        f"  BGP state: {peer['state']}
"
        f"  Local hold timer: {peer['hold_local']}s
"
        f"  Remote hold timer: {peer['hold_remote']}s
"
        f"  Hold timer mismatch: {'YES ← ROOT CAUSE' if peer['hold_local'] != peer['hold_remote'] else 'No'}"
    )

def show_interface(device: str, interface: str) -> str:
    if device not in NETWORK_STATE:
        return f"Error: device {device} not found"
    iface = NETWORK_STATE[device]["interfaces"].get(interface)
    if not iface:
        return f"Interface {interface} not found"
    return (
        f"{interface} is {iface['status']}, line protocol is {iface['status']}
"
        f"  Duplex: {iface['duplex']}
"
        f"  Input errors: {iface['errors']}, CRC: 0"
    )

def show_ip_route(device: str) -> str:
    if device not in NETWORK_STATE:
        return f"Error: device {device} not found"
    routes = NETWORK_STATE[device]["routes"]
    lines = [f"Routing table for {device}:"]
    lines += [f"  B  {r}  [20/0] via 10.1.1.3" for r in routes]
    return "
".join(lines)

def show_logs(device: str, lines: int = 5) -> str:
    if device not in NETWORK_STATE:
        return f"Error: device {device} not found"
    logs = NETWORK_STATE[device]["logs"][-lines:]
    return "
".join(logs) if logs else "No recent log entries"

TOOLS = [
    {
        "name": "ping_device",
        "description": "Ping a target IP from a device to test reachability.",
        "input_schema": {"type": "object", "properties": {
            "device": {"type": "string"}, "target": {"type": "string"},
            "count":  {"type": "integer"},
        }, "required": ["device", "target"]},
    },
    {
        "name": "show_bgp_summary",
        "description": "Show BGP neighbor summary table on a device.",
        "input_schema": {"type": "object", "properties": {
            "device": {"type": "string"},
        }, "required": ["device"]},
    },
    {
        "name": "show_bgp_neighbor_detail",
        "description": "Show detailed BGP neighbor info including timer configuration.",
        "input_schema": {"type": "object", "properties": {
            "device": {"type": "string"}, "neighbor_ip": {"type": "string"},
        }, "required": ["device", "neighbor_ip"]},
    },
    {
        "name": "show_interface",
        "description": "Show interface status, errors, and duplex on a device.",
        "input_schema": {"type": "object", "properties": {
            "device": {"type": "string"}, "interface": {"type": "string"},
        }, "required": ["device", "interface"]},
    },
    {
        "name": "show_ip_route",
        "description": "Show the routing table on a device.",
        "input_schema": {"type": "object", "properties": {
            "device": {"type": "string"},
        }, "required": ["device"]},
    },
    {
        "name": "show_logs",
        "description": "Retrieve recent syslog messages from a device.",
        "input_schema": {"type": "object", "properties": {
            "device": {"type": "string"}, "lines": {"type": "integer"},
        }, "required": ["device"]},
    },
]

TOOL_REGISTRY = {
    "ping_device":              lambda a: ping_device(**a),
    "show_bgp_summary":         lambda a: show_bgp_summary(**a),
    "show_bgp_neighbor_detail": lambda a: show_bgp_neighbor_detail(**a),
    "show_interface":           lambda a: show_interface(**a),
    "show_ip_route":            lambda a: show_ip_route(**a),
    "show_logs":                lambda a: show_logs(**a),
}

# ── ReAct Agent ───────────────────────────────────────────────────────────────
SYSTEM = """You are a senior network engineer AI. Your job is to diagnose network problems autonomously.

For each step:
1. Write a brief Thought explaining your reasoning
2. Call exactly ONE tool to gather information
3. Based on the observation, decide your next step

When you have enough information to identify the root cause, provide a final answer with:
- Root cause
- Evidence
- Recommended fix

Be systematic. Start with layer 3 reachability, then BGP session details, then configuration.
"""

def react_agent(problem: str, max_steps: int = 8):
    print(f"
{'='*60}")
    print(f"  REACT AGENT — NETWORK TROUBLESHOOTER")
    print(f"  Problem: {problem}")
    print(f"{'='*60}")

    messages = [{"role": "user", "content": problem}]
    step = 0

    while step < max_steps:
        step += 1
        response = client.messages.create(
            model=HAIKU,
            max_tokens=600,
            system=SYSTEM,
            tools=TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        print(f"
  ── Step {step} ──────────────────────────────────────")
        tool_called = False

        for block in response.content:
            if block.type == "text" and block.text.strip():
                print(f"  💭 Thought: {block.text.strip()[:200]}")
            elif block.type == "tool_use":
                tool_called = True
                print(f"  ⚡ Action:  {block.name}({json.dumps(block.input)})")
                result = TOOL_REGISTRY[block.name](block.input)
                print(f"  📥 Obs:    {result[:200]}")
                messages.append({
                    "role": "user",
                    "content": [{"type": "tool_result", "tool_use_id": block.id, "content": result}],
                })

        if response.stop_reason == "end_turn" and not tool_called:
            final = next((b.text for b in response.content if b.type == "text"), "")
            print(f"
{'='*60}")
            print(f"  ✅ DIAGNOSIS COMPLETE:")
            print(f"  {final}")
            print(f"{'='*60}")
            return final

    print(f"
  ⚠️  Max steps ({max_steps}) reached — escalating to human")


react_agent("BGP session to peer 10.1.1.2 is down on device core-01. Investigate the root cause.")


---
## Demo 4 — The Four Types of Agent Memory

Memory is what separates an agent from a simple LLM call.
A single API call is **stateless**. An agent **maintains state**.

| Memory Type | What it stores | Networking analogy |
|------------|----------------|-------------------|
| **Working** | Active context window — current task, tool results | RAM in the router |
| **Semantic** | Facts, docs, configs — retrieved via RAG | Secondary storage, loaded via page fault |
| **Episodic** | Past incidents and outcomes | Syslog history, incident tickets |
| **Procedural** | How-to knowledge — runbooks, procedures | Encoded in IOS commands, the router "knows" OSPF |

**The RAG-as-virtual-memory pattern:**
- Context window = main memory (fast, limited)
- Vector database = secondary storage (slow, unlimited)
- Semantic search = the page-fault handler that loads relevant content on demand

This demo implements all four memory types for a network troubleshooting agent.


In [ ]:
# Demo 4: All four agent memory types in action
import anthropic
import json
import datetime
import hashlib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

client = anthropic.Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"

# ────────────────────────────────────────────────────────────────────────────
# 1. WORKING MEMORY — the context window (conversation history)
# ────────────────────────────────────────────────────────────────────────────
class WorkingMemory:
    """Active context window. Limited size — must manage carefully."""

    def __init__(self, max_entries: int = 10):
        self.entries   = []
        self.max_entries = max_entries

    def add(self, role: str, content: str):
        self.entries.append({"role": role, "content": content, "ts": datetime.datetime.now()})
        # Keep only the most recent entries (sliding window)
        if len(self.entries) > self.max_entries:
            self.entries = self.entries[-self.max_entries:]

    def get_messages(self) -> list:
        return [{"role": e["role"], "content": e["content"]} for e in self.entries]

    def summary(self) -> str:
        return f"Working memory: {len(self.entries)}/{self.max_entries} entries"

# ────────────────────────────────────────────────────────────────────────────
# 2. SEMANTIC MEMORY — network documentation (TF-IDF RAG)
# ────────────────────────────────────────────────────────────────────────────
class SemanticMemory:
    """Long-term knowledge base — retrieved on demand via similarity search."""

    DOCS = [
        "BGP hold timer mismatch: local and remote peers must agree on hold time. Fix: set matching timers with 'neighbor X timers keepalive holdtime'.",
        "BGP state IDLE means the session never established. Check: ACLs on port 179, AS number configuration, and hold timer values.",
        "OSPF MTU mismatch causes EXSTART state. Both sides must have matching IP MTU or use 'ip ospf mtu-ignore'.",
        "OSPF neighbors stuck in INIT: hello packets reach the peer but replies don't return. Check ACLs, multicast group membership.",
        "Interface CRC errors indicate physical layer issues: bad cable, SFP failure, or duplex mismatch. Use 'duplex full' on both sides.",
        "BGP authentication: if MD5 is configured on one side, it must match on the other. Error shows as TCP RST.",
    ]

    def __init__(self):
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.matrix     = self.vectorizer.fit_transform(self.DOCS)

    def retrieve(self, query: str, top_k: int = 2) -> str:
        q_vec  = self.vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self.matrix).flatten()
        top_idx = np.argsort(-scores)[:top_k]
        results = [self.DOCS[i] for i in top_idx if scores[i] > 0.05]
        return "
".join(results) if results else "No relevant docs found."

# ────────────────────────────────────────────────────────────────────────────
# 3. EPISODIC MEMORY — incident history (past events and outcomes)
# ────────────────────────────────────────────────────────────────────────────
class EpisodicMemory:
    """Historical incidents — lets the agent learn from past experience."""

    def __init__(self):
        self.incidents = []

    def record(self, problem: str, root_cause: str, fix: str):
        self.incidents.append({
            "id":         hashlib.md5(problem.encode()).hexdigest()[:8],
            "ts":         datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
            "problem":    problem,
            "root_cause": root_cause,
            "fix":        fix,
        })

    def search(self, query: str) -> str:
        """Simple keyword search through past incidents."""
        query_lower = query.lower()
        matches = [
            inc for inc in self.incidents
            if any(w in inc["problem"].lower() or w in inc["root_cause"].lower()
                   for w in query_lower.split() if len(w) > 3)
        ]
        if not matches:
            return "No similar past incidents found."
        result = []
        for inc in matches[-3:]:  # last 3 matches
            result.append(
                f"[{inc['ts']}] Problem: {inc['problem']}
"
                f"  Root cause: {inc['root_cause']}
"
                f"  Fix applied: {inc['fix']}"
            )
        return "
".join(result)

# ────────────────────────────────────────────────────────────────────────────
# 4. PROCEDURAL MEMORY — how-to knowledge (encoded in system prompt)
# ────────────────────────────────────────────────────────────────────────────
PROCEDURAL_MEMORY = """Standard BGP Diagnostic Procedure:
1. Verify TCP reachability on port 179 (ping the peer IP)
2. Check BGP summary: show bgp summary
3. Check neighbor detail for state and timer config
4. Review syslog for BGP-related messages
5. Cross-reference with documentation
6. Produce root-cause analysis with fix recommendation

Always check the simplest layer first (IP connectivity) before BGP-specific issues.
"""

# ────────────────────────────────────────────────────────────────────────────
# DEMO: Agent uses all four memory types
# ────────────────────────────────────────────────────────────────────────────
working   = WorkingMemory(max_entries=10)
semantic  = SemanticMemory()
episodic  = EpisodicMemory()

# Pre-populate episodic memory with past incidents
episodic.record(
    problem="BGP peer 10.2.2.2 went down — AS 65004",
    root_cause="Hold timer mismatch: local=90s, remote=180s",
    fix="Applied: neighbor 10.2.2.2 timers 60 180 on core-02",
)
episodic.record(
    problem="OSPF adjacency between core-01 and core-03 stuck in EXSTART",
    root_cause="MTU mismatch: core-01 MTU=1500, core-03 MTU=9000 (jumbo frames)",
    fix="Applied ip ospf mtu-ignore on both sides",
)

# Current problem
problem = "BGP session to 10.1.1.2 (AS 65002) is down on core-01"
print(f"{'='*60}")
print(f"  MEMORY-AWARE AGENT DEMO")
print(f"  Problem: {problem}")
print(f"{'='*60}
")

# Agent retrieves from all memory types to compose context
print("  📂 SEMANTIC MEMORY — retrieving relevant docs:")
docs = semantic.retrieve(problem)
print(f"  {docs[:200]}
")

print("  📚 EPISODIC MEMORY — searching past incidents:")
history = episodic.search(problem)
print(f"  {history[:200]}
")

print("  📋 PROCEDURAL MEMORY — standard procedure:")
print(f"  {PROCEDURAL_MEMORY[:200]}...
")

# Build augmented prompt combining all memory types
augmented_prompt = f"""Current problem: {problem}

Relevant documentation:
{docs}

Similar past incidents:
{history}

Standard diagnostic procedure:
{PROCEDURAL_MEMORY}

Based on past incidents and documentation, what is the most likely root cause
and what specific command should we check first?
"""

working.add("user", augmented_prompt)

response = client.messages.create(
    model=HAIKU,
    max_tokens=300,
    messages=working.get_messages(),
)
answer = response.content[0].text
working.add("assistant", answer)

print("  🧠 AGENT RESPONSE (using all 4 memory types):")
print(f"  {answer}")
print(f"
  {working.summary()}")


---
## Demo 5 — Multi-Agent: Manager Routes to Specialist Agents

When problems are complex, a single agent reaches its limits.
**Multi-agent systems** coordinate specialized agents — each expert in its domain.

**The Manager/Supervisor pattern:**
```
                    ┌─────────────┐
                    │   MANAGER   │   ← receives the problem
                    │   AGENT     │   ← classifies it
                    └──────┬──────┘   ← dispatches to specialist
                           │
              ┌────────────┼────────────┐
              ▼            ▼            ▼
        ┌──────────┐ ┌──────────┐ ┌──────────┐
        │   BGP    │ │  OSPF    │ │Interface │
        │SPECIALIST│ │SPECIALIST│ │SPECIALIST│
        └──────────┘ └──────────┘ └──────────┘
```

**Networking analogy:** The Manager is like your NOC duty manager.
They receive alerts, classify them (BGP issue? OSPF issue? Physical?), and route
to the right specialist team. The specialists don't handle everything — they focus.

This mirrors the **Actor-Critic** pattern too:
- Specialist Agent = Actor (investigates and proposes a fix)
- Manager Agent = Critic (validates the diagnosis before escalating)


In [ ]:
# Demo 5: Multi-agent system — Manager + 3 Specialist agents
import anthropic
import json

client = anthropic.Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

# ── Specialist knowledge bases ───────────────────────────────────────────────
SPECIALIST_KNOWLEDGE = {
    "bgp": {
        "system": (
            "You are a BGP specialist. You diagnose BGP routing issues. "
            "Focus on: session states, timer mismatches, AS configuration, "
            "prefix filtering, route reflection, and BGP authentication. "
            "Give a specific diagnosis with the exact fix command."
        ),
        "context": (
            "Available diagnostic data:
"
            "- core-01 BGP neighbor 10.1.1.2 (AS 65002): state=IDLE
"
            "- Local hold timer: 90s  Remote hold timer: 180s  ← MISMATCH
"
            "- Syslog: BGP-5-ADJCHANGE: neighbor 10.1.1.2 Down -- Hold Timer Expired
"
            "- Ping to 10.1.1.2: SUCCESS (layer 3 is fine)"
        ),
    },
    "ospf": {
        "system": (
            "You are an OSPF specialist. You diagnose OSPF routing issues. "
            "Focus on: adjacency states, area configuration, LSA types, "
            "MTU mismatches, hello/dead interval mismatches, and redistribution. "
            "Give a specific diagnosis with the exact fix command."
        ),
        "context": (
            "Available diagnostic data:
"
            "- core-01 OSPF neighbors: 10.2.2.2=FULL, 10.2.2.3=FULL  (all healthy)
"
            "- No OSPF-related syslog messages in last 24h
"
            "- Area 0 backbone intact"
        ),
    },
    "interface": {
        "system": (
            "You are a network interface specialist. You diagnose physical and "
            "data-link layer issues. Focus on: interface status, error counters, "
            "duplex/speed mismatches, SFP issues, and cable problems. "
            "Give a specific diagnosis with the exact fix command."
        ),
        "context": (
            "Available diagnostic data:
"
            "- GigabitEthernet0/0: up/up, errors=0, duplex=full
"
            "- GigabitEthernet0/1: up/up, errors=0, duplex=full
"
            "- No CRC errors, no input/output drops"
        ),
    },
}

def specialist_agent(domain: str, problem: str) -> str:
    """A specialized agent that investigates one domain."""
    spec = SPECIALIST_KNOWLEDGE[domain]
    prompt = f"{spec['context']}

Problem to investigate: {problem}"

    response = client.messages.create(
        model=HAIKU,
        max_tokens=300,
        system=spec["system"],
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text.strip()

# ── Manager Agent ─────────────────────────────────────────────────────────────
MANAGER_SYSTEM = """You are a NOC manager AI. Your job is to:
1. Classify the incoming network problem
2. Route it to the right specialist: "bgp", "ospf", or "interface"
3. After receiving the specialist's diagnosis, validate it and produce a final report

When classifying, respond with JSON:
{"domain": "bgp"|"ospf"|"interface", "reason": "one sentence why"}

When producing final report, structure it as:
DOMAIN: ...
ROOT CAUSE: ...
FIX: ...
RISK LEVEL: low|medium|high
"""

def manager_agent(problem: str) -> dict:
    """Manager that classifies the problem and dispatches to specialists."""
    print(f"
{'='*60}")
    print(f"  MULTI-AGENT SYSTEM")
    print(f"  Incoming problem: {problem}")
    print(f"{'='*60}")

    # Step 1: Manager classifies the problem
    print("
  🎯 MANAGER AGENT — classifying problem...")
    classify_response = client.messages.create(
        model=HAIKU,
        max_tokens=100,
        system=MANAGER_SYSTEM,
        messages=[{"role": "user", "content": f"Classify this problem: {problem}"}],
    )
    raw = classify_response.content[0].text.strip()

    # Parse classification
    try:
        # Extract JSON from response
        import re
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        classification = json.loads(json_match.group()) if json_match else {"domain": "bgp", "reason": "default"}
    except Exception:
        classification = {"domain": "bgp", "reason": "classification failed, defaulting to BGP"}

    domain = classification.get("domain", "bgp")
    reason = classification.get("reason", "")
    print(f"  → Classified as: [{domain.upper()}] — {reason}")

    # Step 2: Dispatch to specialist
    print(f"
  🔬 {domain.upper()} SPECIALIST AGENT — investigating...")
    specialist_result = specialist_agent(domain, problem)
    print(f"  Specialist report:
    {specialist_result[:300]}")

    # Step 3: Manager validates and produces final report
    print(f"
  ✅ MANAGER AGENT — final validation...")
    final_response = client.messages.create(
        model=HAIKU,
        max_tokens=250,
        system=MANAGER_SYSTEM,
        messages=[{
            "role": "user",
            "content": (
                f"Problem: {problem}

"
                f"Specialist ({domain}) diagnosis:
{specialist_result}

"
                f"Produce the final report."
            ),
        }],
    )
    final_report = final_response.content[0].text.strip()

    print(f"
  📋 FINAL REPORT:")
    print(f"  {final_report}")

    return {
        "problem":    problem,
        "domain":     domain,
        "specialist": specialist_result,
        "report":     final_report,
    }


# ── Run the multi-agent system on 2 different problems ────────────────────────
problems = [
    "BGP session to peer 10.1.1.2 is down on core-01 — BGP neighbor shows IDLE state",
    "Network is showing interface errors and high CRC count on core-01",
]

for prob in problems:
    result = manager_agent(prob)
    print()


---
## Summary — What You Built

You went from a simple loop visualization all the way to a production-pattern multi-agent system:

### Demo progression:
1. **Agent Loop** — visualized the 4-phase Perception → Reason → Act → Update cycle
2. **Tool Calling** — saw exactly how the protocol works: tool schema → tool_use → tool_result
3. **ReAct Agent** — 6 real network tools, full thought-action-observation loop
4. **Memory Architecture** — all 4 types: Working / Semantic / Episodic / Procedural
5. **Multi-Agent** — Manager classifies → Specialist investigates → Manager validates

### Key principles to remember:

**The LLM never executes code directly.**
It generates a structured `tool_use` block. Your Python code runs it.
This separation = where all your safety controls must live.

**Always set a max_steps / max_iter limit.**
Without it, a stuck agent loops forever. Iteration limits are your circuit breaker.

**Graduated autonomy: Crawl → Walk → Run**
- Crawl: every action requires human approval
- Walk: read-only tools are automatic, changes need approval
- Run: bounded autonomous operation with async human oversight

**The 5 agent autonomy levels:**
Level 1 (display info) → Level 2 (route workflows) → Level 3 (read-only tools)
→ Level 4 (multi-step with approval gates) → Level 5 (fully autonomous)

Start at Level 3 for network operations. That alone delivers enormous value.

---
*Chapter 15 — Building AI Agents | AI for Networking and Security Engineers*
